In [2]:
import scanpy as sc
import pandas as pd
import os
for dataset_number in range(1,32):
    dataset_number = str(dataset_number)
    adata =sc.read_h5ad(f"/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset{dataset_number}/scRNA.h5ad")
    adata.obs['cell_type']=adata.obs['celltype_final']
    adata.write_h5ad(f"/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset{dataset_number}/scRNA.h5ad")
    adata_st = sc.read_h5ad(f"/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset{dataset_number}/Spatial.h5ad")
    st_label = adata_st.uns['density']
    st_label_df = pd.DataFrame(st_label)
    os.makedirs(f"/mnt/d/ST_Graduation_Project/SC_MAP_ST/evaluate/Data{dataset_number}", exist_ok=True)
    st_label_df.to_csv(f"/mnt/d/ST_Graduation_Project/SC_MAP_ST/evaluate/Data{dataset_number}/dataset{dataset_number}_density.csv")

In [1]:
import os
import time
import sys

# 获取当前环境的 Python 路径
current_python = sys.executable 

for i in range(1, 33):
    dataset_number = str(i)
    
    print(f"\n{'='*60}")
    print(f"🔥 正在处理 Dataset {dataset_number} / 32 ...")
    print(f"{'='*60}\n")
    
    # ================= 定义路径 =================
    sc_file = f"/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset{dataset_number}/scRNA.h5ad"
    st_file = f"/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset{dataset_number}/Spatial.h5ad"
    out_dir = f"../deconv_results/DATA{dataset_number}"
    model_path = f"{out_dir}/final_vae.pth"

    # ================= 1. 运行 Stage 1 =================
    cmd1 = (
        f"{current_python} ../stage1.py "
        f"--sc_file {sc_file} "
        f"--st_file {st_file} "
        f"--n_epochs 150 "
        f"--output_dir {out_dir}"
    )
    
    print(f"Running Stage 1...")
    exit_code1 = os.system(cmd1)
    
    if exit_code1 != 0:
        print(f"\n❌ 严重错误：Dataset {dataset_number} 在 Stage 1 失败！")
        print("⛔️ 正在终止整个循环，请检查问题后再继续。")
        break  # <--- 这里改成了 break，一出错就全停
    
    # ================= 2. 运行 Stage 2 =================
    cmd2 = (
        f"{current_python} ../stage2.py "
        f"--st_file {st_file} "
        f"--stage1_model_path {model_path} "
        f"--output_dir {out_dir}/ "
        f"--n_epoch 300"
    )
    
    print(f"Running Stage 2...")
    exit_code2 = os.system(cmd2)

    if exit_code2 != 0:
        print(f"\n❌ 严重错误：Dataset {dataset_number} 在 Stage 2 失败！")
        print("⛔️ 正在终止整个循环，请检查问题后再继续。")
        break # <--- 这里也改成了 break
        
    print(f"✅ Dataset {dataset_number} 全部完成！")
    time.sleep(2)


🔥 正在处理 Dataset 1 / 32 ...

Running Stage 1...
Using device: cpu
Stage 1 Training: VAE (SC + ST, Marker Genes)
Configuration:
   Marker genes per type: 100
   Leiden resolution: 4
   Batch size: 512
   Epochs: 150
   Learning rate: 0.0005
   Beta (KL weight): 0.1
   Hidden dims: [512, 256]
   Latent dim: 128
   Loss type: MSE
   Lambda MMD: 0.1
   Dual Decoder: True
   Aggregation method: mean
Loading datasets...
   SC shape: (10000, 33691)
   ST shape: (1000, 46732)
   Common genes: 32897
   SC final: (10000, 32897)
   ST final: (1000, 32897)
Computing clusters and marker genes...
Total: 1980 marker genes
   SC data: (10000, 1980)
   Number of clusters: 79

❌ 严重错误：Dataset 1 在 Stage 1 失败！
⛔️ 正在终止整个循环，请检查问题后再继续。


Killed
